In [1]:
import psycopg2
import pandas as pd

In [2]:
# environment variable can take two values: DEV (development), PRD (production)
env = 'DEV'

In [3]:
# for production we will use following database uri variable
DATABASE_URI = ''

In [4]:
try:
    if env == 'DEV':
        con = psycopg2.connect(host="localhost", user='postgres', database='value-investing-dev', port='5432', password='v,1846PSVv,1846PSV')
    elif env == 'PRD':
        con = psycopg2.connect(DATABASE_URI)

    #create cursor to execute sql statements
    cur = con.cursor()

    # read the tax rates from the damodaran file
    df = pd.read_csv('CorporateTaxRate.csv')

    # iterate through dataframe to create new entries or update existing ones
    for index, row in df.iterrows():
        # print(f'Country: {row["Country"]}, TaxRate: {row["Corporate Tax Rate"]}')
        country = str(row['Country'])

        # extract tax rate; check if % sign is included
        tax_rate = row["Corporate Tax Rate"]
        if '%' in tax_rate:
            tax_rate = float(tax_rate.split('%')[0])/100
        else:
            tax_rate = float(tax_rate)/100

        # first we will check if already a country with the same name exists in the database
        sql_select_query = """select * from public.dcf_taxrates where lower(country) = %s"""

        # execute the sql query
        cur.execute(sql_select_query, (country.lower(),))

        countries = cur.fetchall()

        # if length is zero (so there does not yet exist and entry in the table), we will create a new entry; otherwise we will update the existing one
        if len(countries) == 0:
            sql_insert_query = """INSERT INTO public.dcf_taxrates(country, "taxRate") VALUES(%s, %s)"""
            cur.execute(sql_insert_query, (country, tax_rate))
            print(f'inserted into table: Country: {country}, Tax Rate {tax_rate}')
        else:
            sql_update_query = """UPDATE public.dcf_taxrates SET "taxRate"=%s where lower(country)=%s"""
            cur.execute(sql_update_query, (tax_rate, country.lower()))
            print(f'updated in table: Country: {country}, Tax Rate {tax_rate}')
        
        # commit
        con.commit()
    
    cur.close()

except Exception as error:
    print('Could not connect to the database: ', error)


updated in table: Country: Abu Dhabi, Tax Rate 0.15
updated in table: Country: Albania, Tax Rate 0.15
updated in table: Country: Algeria, Tax Rate 0.26
updated in table: Country: Andorra (Principality of), Tax Rate 0.1898
updated in table: Country: Angola, Tax Rate 0.25
updated in table: Country: Anguilla, Tax Rate 0.2563
updated in table: Country: Antigua & Barbuda, Tax Rate 0.2563
updated in table: Country: Argentina, Tax Rate 0.35
updated in table: Country: Armenia, Tax Rate 0.18
updated in table: Country: Aruba, Tax Rate 0.25
updated in table: Country: Australia, Tax Rate 0.3
updated in table: Country: Austria, Tax Rate 0.24
updated in table: Country: Azerbaijan, Tax Rate 0.2
updated in table: Country: Bahamas, Tax Rate 0.0
updated in table: Country: Bahrain, Tax Rate 0.0
updated in table: Country: Bangladesh, Tax Rate 0.325
updated in table: Country: Barbados, Tax Rate 0.055
updated in table: Country: Belarus, Tax Rate 0.18
updated in table: Country: Belgium, Tax Rate 0.25
updated